In [1]:
#Import Libraries
import pandas as pd
import numpy as np

import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.figure_factory as ff
from matplotlib import pyplot as plt

##Cross-Validation using KFold
from numpy import array
from sklearn.model_selection import KFold

##Scaler
from sklearn.decomposition import PCA
from sklearn import preprocessing
from sklearn.preprocessing import MinMaxScaler

In [85]:
# Ingest Data
df = pd.read_csv('titanictrain.csv')
origdata = pd.read_csv('titanictrain.csv')
train_test_data = [df]

In [88]:
# Variables for preprocessing features
## Simplify Title Mapping
TitleMap = {
    ' Mr': 'Mr', ' Mrs':'Mrs', ' Miss':'Miss', ' Master':'Master', ' Don':'Other', ' Dona':'Mrs',' Rev':'Other', ' Dr':'Other', 
    ' Mme':'Mrs',' Ms':'Miss', ' Major':'Other', ' Lady':'Mrs', ' Sir':'Other', ' Mlle':'Miss', ' Col':'Other',
    ' Capt':'Other',' the Countess':'Mrs', ' Jonkheer':'Other' 
}

#Process Age with bins.
#Create Bin Labels for Amounts (lowest to highest in funding)
AmtBinLabel = ['Child', 'Teen', 'Adult', 'MiddleAge', 'Senior']
#Binning Amounts
AmtBins = [0, 12, 19, 35, 65, 100]

#Travel Bins
TravelBinLabel = ['Alone','Couple','Small','Large']
TravelBins = [0, 1, 2, 5, 100]

#Process features and then label encode
le = preprocessing.LabelEncoder()
for dataset in train_test_data:
    # Processing Name to strip Title.  Done first since we use Title to group by Age and resolve NANs.
    dataset['Title'] = dataset['Name'].str.split('.').str[0].str.split(',').str[1]
    # Process NANs
    dataset['Title'] = dataset['Title'].map(TitleMap)
    dataset['Age'].fillna(dataset.groupby('Title')['Age'].transform('median'), inplace=True)
    dataset['AgeBin1'] = pd.cut(dataset['Age'], bins=AmtBins, labels=AmtBinLabel, precision=0)
    dataset['Embarked'].fillna('S', inplace=True) #Majority seems to be S.
    dataset["Fare"].fillna(dataset.groupby("Pclass")["Fare"].transform("median"), inplace=True)
    #Calculate group size based on similar Ticket values instead of Name+Cabin.
    dataset['TravelSize'] = dataset['SibSp'] + dataset['Parch'] + 1 #Group size calculation.
    dataset['TicketGroup'] = dataset['TravelSize'].groupby(dataset['Ticket']).transform('sum') #Similar tickets.
    dataset['TravelType'] = pd.cut(dataset['TicketGroup'], bins=TravelBins, labels=TravelBinLabel, precision=0)
    ## Scale Fare and Age using MinMax 
    mms = MinMaxScaler()
    dataset['Fare'] = mms.fit_transform(pd.DataFrame(dataset['Fare']))
    dataset['Age'] = mms.fit_transform(pd.DataFrame(dataset['Age']))
    dataset['AgeBin1'] = le.fit_transform(dataset['AgeBin1'])
    dataset['Pclass'] = le.fit_transform(dataset['Pclass'])
    dataset['Sex'] = le.fit_transform(dataset['Sex'])
    dataset['Embarked'] = le.fit_transform(dataset['Embarked'])
    dataset['Title'] = le.fit_transform(dataset['Title'])
    dataset['TravelType'] = le.fit_transform(dataset['TravelType'])
    ## Drop Cabin (too many NANs) and other already used features.
    dataset.drop(['Cabin'], axis=1, inplace=True)
    dataset.drop(['Age'], axis=1, inplace=True)
    dataset.drop(['PassengerId'], axis=1, inplace=True)
    dataset.drop(['Name'], axis=1, inplace=True)
    dataset.drop(['Ticket'], axis=1, inplace=True)
    dataset.drop(['TravelSize'], axis=1, inplace=True)
    dataset.drop(['TicketGroup'], axis=1, inplace=True)
    dataset.drop(['SibSp'], axis=1, inplace=True) # Age + Title is preferred.  
    dataset.drop(['Parch'], axis=1, inplace=True) # Could group by Family size if needed.
    dataset.drop(['Embarked'], axis=1, inplace=True) #Drop to see if PC improves.
    dataset.drop(['Pclass'], axis=1, inplace=True) #Drop to see if PC improves.

In [89]:
Attributes = 5
PCList = []
for i in range(1,Attributes+1):
    PCList.append("PC"+str(i))

DimensionsList = list(df.iloc[:,1:6].columns)

y = df.Survived  #dependent/label variable
df.drop(['Survived'], axis=1, inplace=True)
x = df  #attribute variables

In [90]:
#Dendrogram Groups
figdend = ff.create_dendrogram(x.corr(), labels=DimensionsList)
figdend.update_layout(width=1000, height=300)
figdend.show()

In [91]:
xscale = pd.DataFrame(MinMaxScaler().fit_transform(x))
xscale.columns = DimensionsList
pca = PCA(n_components = Attributes)
principalComponents = pca.fit_transform(xscale)
principalDataframe = pd.DataFrame(data = principalComponents, columns = PCList)
newDataframe = pd.concat([principalDataframe, y],axis = 1)

In [92]:
#Factor Loadings.  Look for values closer to +1 and -1.  
#Then look for orthogonality between Factors.
#This way the data will cluster on two separate axes more clearly.
Loadings = pd.DataFrame(pca.components_.T * np.sqrt(pca.explained_variance_), columns = PCList)
Loadings.insert(0, "Attribute", DimensionsList)
Loadings.style.background_gradient(cmap='inferno')

,Attribute,PC1,PC2,PC3,PC4,PC5
0,Sex,-0.439087,0.181776,-0.050966,-0.006052,0.000432
1,Fare,0.033334,0.030215,-0.012847,0.000090,0.084964
2,Title,-0.013950,0.007610,0.005530,0.195245,0.000010
3,AgeBin1,0.053054,0.157350,0.291384,-0.004251,-0.000764
4,TravelType,0.269622,0.261723,-0.138462,0.001071,-0.009650


In [93]:
#Collect the Eigenvalues.
Eigenvalues = pd.DataFrame(pca.explained_variance_)
Eigenvalues = Eigenvalues.T
Eigenvalues.columns = PCList
Eigenvalues

,PC1,PC2,PC3,PC4,PC5
0,0.269614,0.127271,0.10687,0.038177,0.007313


In [94]:
variance = pca.explained_variance_ratio_ #calculate variance ratios
vardf = pd.DataFrame(variance, columns=['Variance']) #cumulative sum of variance explained with [n] features
vardf['PrincipalComponent'] = PCList
vardf['CumulativeVariance'] = vardf['Variance'].cumsum()

In [95]:
#View the covariance matrix.  This shows how the attributes move together (degree of correlation)
covdf = pd.DataFrame(pca.get_covariance())
covdf.columns = PCList
covdf.insert(0, 'Dimensions',DimensionsList)
covdf.style.background_gradient(cmap='inferno')

,Dimensions,PC1,PC2,PC3,PC4,PC5
0,Sex,0.228475,-0.008453,0.006045,-0.009518,-0.063766
1,Fare,-0.008453,0.009408,-0.000288,0.002714,0.017855
2,Title,0.006045,-0.000288,0.038404,0.001239,-0.002326
3,AgeBin1,-0.009518,0.002714,0.001239,0.112497,0.015144
4,TravelType,-0.063766,0.017855,-0.002326,0.015144,0.160461


In [29]:
#View the Scree plot to identify the elbow in the curve based on the cumulative scores.
fig = make_subplots(rows=2, cols=1)
fig.add_trace(
    go.Bar(x=vardf['PrincipalComponent'], 
           y=vardf['Variance']), row=1, col=1,
)

fig.add_trace(
    go.Scatter(x=vardf['PrincipalComponent'], 
               y=vardf['CumulativeVariance']), row=2, col=1
)

fig.update_layout(height=500, width=800, title_text="Individual & Cumulative Scree Plots")
fig.show()

In [46]:
fig = px.scatter_matrix(
    Loadings,
    dimensions=['PC1', 'PC2','PC3', 'PC4'],
    color="Attribute")
fig.show()

In [19]:
#PC2 and PC4 illustrate Sex, TravelType, Fare, and AgeBin1 features cluster and positive quadrants.
fig = px.scatter_matrix(
    Loadings,
    dimensions=['PC1', 'PC2'],
    color="Attribute")
fig.show()

In [18]:
#  PC1-PC4 are give the highest returns (93%).
#  Here the cluster look helpful.
fig1 = px.scatter(
                x = newDataframe.PC1,
                y = newDataframe.PC2,
                color = newDataframe.Survived,
                marginal_y="rug", 
                marginal_x="rug")
fig1.show()

In [115]:
fulldata = pd.concat([newDataframe, df],axis = 1)
fulldata = pd.concat([fulldata, origdata.Name],axis = 1)

In [38]:
#  PC1-PC4 are give the highest returns (93%).
#  Here the clusters are clear.
fig1 = px.scatter(
                x = fulldata.PC1,
                y = fulldata.Survived,
                color = fulldata.Sex,
                marginal_y="rug", 
                marginal_x="rug")
fig1.show()

In [114]:
# Finally, let's see the types of Males most likely to perish.
test = fulldata[fulldata.Sex==1]
fig = px.scatter_3d(test, x='PC1', y='TravelType', z='AgeBin1',
              color='Survived', size='Fare', size_max=50,
              symbol='Sex', hover_name='Name', opacity=0.0)

# tight layout
fig.update_layout(margin=dict(l=0, r=0, b=0, t=0))